# Outcome Analysis — Fine-tuned LLM vs. BG Oracle

Benchmarks the fine-tuned Llama-3-8B market-making policy against the Bergault–Guéant
quadratic-approximation oracle on **held-out CUSIPs**, with the belief-optimal and random
policies as reference points.

**Sections**
1. Setup & trace collection
2. PnL / BG-objective distributions (oracle vs LLM vs ICL vs belief vs random) — *BEGV Figs 12–13 analogue*
3. Optimality-gap decomposition (the headline number)
4. Action agreement & confusion matrix — *where does the LLM diverge?*
5. Quote-vs-state curves (skew vs inventory, skew vs regime) — *did it recover the BG structure?*
6. Regime-inference quality (action correctness vs belief confidence)

> The LLM cells require the fine-tuned adapter + Llama-3-8B + torch (run on the GPU box).
> Cells for oracle / belief / random run anywhere. Set `HAVE_LLM = True` once the adapter is available.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))   # repo root so `import ftp...` works
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from rfqsim.config import SimConfig
from rfqsim.mmpp import build_generator
from ftp.pipeline import GenConfig
from ftp.trace import collect_traces
from ftp.rollout import OraclePolicy, RandomPolicy, pooled_gap
from ftp.belief import BeliefOptimalPolicy, MMPPBeliefFilter

cfg = SimConfig()
gc  = GenConfig(horizon_days=1.0)
REGIME = {0:'LL', 1:'LH', 2:'HL', 3:'HH'}
N_EVAL_CUSIPS = 30      # bump up for final figures
EPISODES_PER  = 4

# Toggle once the fine-tuned adapter is present on this machine.
HAVE_LLM = False
ADAPTER_PATH = '../ckpt/final'


## 1. Trace collection

`collect_traces` rolls a policy and the oracle in lockstep on identical RFQ streams, logging per-decision actions, regime, belief, inventory, and pnl. We collect one trace set per policy.

In [ ]:
# --- policies that run without the LLM ---
def random_factory(oracle, cfg):
    return RandomPolicy(np.random.default_rng(np.random.randint(1<<30)))

def belief_factory(oracle, cfg):
    Q = build_generator(cfg.mmpp)
    lo = cfg.mmpp.lam_low_frac*gc.sector_intensity
    hi = cfg.mmpp.lam_high_frac*gc.sector_intensity
    bf = MMPPBeliefFilter(Q, np.array([lo,lo,hi,hi]), np.array([lo,hi,lo,hi]))
    return BeliefOptimalPolicy(oracle, bf)

# same seed => same held-out CUSIPs/streams across policies (paired comparison)
SEED = 999
rows_belief, obj_belief = collect_traces(cfg, gc, belief_factory,
                                         n_cusips=N_EVAL_CUSIPS, episodes_per=EPISODES_PER, seed=SEED)
rows_random, obj_random = collect_traces(cfg, gc, random_factory,
                                         n_cusips=N_EVAL_CUSIPS, episodes_per=EPISODES_PER, seed=SEED)
df_belief = pd.DataFrame(rows_belief)
df_random = pd.DataFrame(rows_random)
print('belief decisions:', len(df_belief), '| random decisions:', len(df_random))
df_belief.head()


In [ ]:
# --- LLM trace (GPU box only) ---
if HAVE_LLM:
    from ftp.llm_policy import LLMPolicy, ICLPolicy, build_support_prefix, load_sft_model
    from ftp.sft_data import load_jsonl
    model, tok = load_sft_model(ADAPTER_PATH)

    def llm_factory(oracle, cfg):
        p = LLMPolicy(model, tok, z=gc.z); p.reset(); return p

    rows_llm, obj_llm = collect_traces(cfg, gc, llm_factory,
                                       n_cusips=N_EVAL_CUSIPS, episodes_per=EPISODES_PER, seed=SEED)
    df_llm = pd.DataFrame(rows_llm)

    # optional ICL baseline (few-shot, no fine-tuning)
    support = build_support_prefix(load_jsonl('/mnt/user-data/outputs/sft_data/train.jsonl'), k=2)
    def icl_factory(oracle, cfg):
        p = ICLPolicy(model, tok, support, z=gc.z); p.reset(); return p
    rows_icl, obj_icl = collect_traces(cfg, gc, icl_factory,
                                       n_cusips=N_EVAL_CUSIPS, episodes_per=EPISODES_PER, seed=SEED)
    df_icl = pd.DataFrame(rows_icl)
    print('LLM decisions:', len(df_llm))
else:
    df_llm = obj_llm = df_icl = obj_icl = None
    print('HAVE_LLM=False — skipping LLM traces. Oracle/belief/random analyses still run.')


## 2. BG-objective distributions

*BEGV Figs 12–13 analogue.* Per-episode realized BG objective for each policy. Shows not just the mean but the **spread** — does the LLM match the oracle's risk profile, or occasionally blow up?

In [ ]:
def ep_obj_frame(obj_list, name):
    d = pd.DataFrame(obj_list)
    return pd.DataFrame({'episode': d['episode'], 'obj': d['obj_policy'], 'policy': name})

# oracle objective is logged identically in every trace set (obj_oracle column)
oracle_obj = pd.DataFrame(obj_belief)[['episode','obj_oracle']].rename(columns={'obj_oracle':'obj'})
oracle_obj['policy'] = 'oracle'

frames = [oracle_obj,
          ep_obj_frame(obj_belief, 'belief-opt'),
          ep_obj_frame(obj_random, 'random')]
if obj_llm is not None: frames.append(ep_obj_frame(obj_llm, 'SFT LLM'))
if obj_icl is not None: frames.append(ep_obj_frame(obj_icl, 'ICL'))
allobj = pd.concat(frames, ignore_index=True)

fig, ax = plt.subplots(figsize=(9,5))
order = [p for p in ['oracle','SFT LLM','ICL','belief-opt','random'] if p in allobj['policy'].unique()]
for p in order:
    v = allobj[allobj.policy==p]['obj']
    ax.hist(v, bins=30, alpha=0.5, label=f'{p} (μ={v.mean():.3f})')
ax.set_xlabel('per-episode BG objective'); ax.set_ylabel('count')
ax.set_title('BG-objective distribution by policy (held-out CUSIPs)'); ax.legend()
plt.tight_layout(); plt.show()


## 3. Optimality-gap decomposition (headline)

Pooled gap = (ΣOPT − Σpolicy)/(ΣOPT − Σrandom). 0 = matches oracle, 1 = random.

In [ ]:
O = pd.DataFrame(obj_belief)['obj_oracle'].values
R = pd.DataFrame(obj_random)['obj_policy'].values
gaps = {'full-info oracle': 0.0,
        'random': pooled_gap(R, O, R),
        'belief-optimal': pooled_gap(pd.DataFrame(obj_belief)['obj_policy'].values, O, R)}
if obj_llm is not None:
    gaps['SFT LLM'] = pooled_gap(pd.DataFrame(obj_llm)['obj_policy'].values, O, R)
if obj_icl is not None:
    gaps['ICL'] = pooled_gap(pd.DataFrame(obj_icl)['obj_policy'].values, O, R)

gdf = pd.DataFrame({'policy': list(gaps), 'pooled_gap': list(gaps.values())})
display(gdf)
fig, ax = plt.subplots(figsize=(7,4))
ax.bar(gdf['policy'], gdf['pooled_gap'], color=['#2a9d8f' if p=='SFT LLM' else '#bbb' for p in gdf['policy']])
ax.axhline(0, color='k', lw=0.8); ax.set_ylabel('pooled optimality gap')
ax.set_title('Optimality gap vs full-info oracle (lower = better)')
plt.xticks(rotation=20); plt.tight_layout(); plt.show()


## 4. Action agreement & confusion matrix

*Where does the policy diverge?* Exact-match and off-by-one rates overall and **conditioned on the true regime** — this localizes failures. The confusion matrix over the 9 offset bins shows systematic bias (e.g. under-skewing).

In [ ]:
def agreement_report(df, name):
    overall = df['match'].mean()
    offby1 = (df['absdiff']<=1).mean()
    by_reg = df.groupby('regime')['match'].mean().rename(index=REGIME)
    print(f'--- {name} ---')
    print(f'exact-match={overall:.1%}  within-1-bin={offby1:.1%}')
    print('exact-match by regime:', {k: f'{v:.1%}' for k,v in by_reg.items()})
    return by_reg

reg_b = agreement_report(df_belief, 'belief-optimal')
if df_llm is not None:
    reg_l = agreement_report(df_llm, 'SFT LLM')


In [ ]:
# Confusion matrix over action bins for the policy under analysis
from ftp.serialize import N_ACTIONS
def confusion(df, title):
    M = np.zeros((N_ACTIONS, N_ACTIONS), dtype=int)
    for ao, ap in zip(df['a_oracle'], df['a_policy']):
        M[ao, ap] += 1
    Mn = M / M.sum(axis=1, keepdims=True).clip(min=1)
    fig, ax = plt.subplots(figsize=(6,5))
    im = ax.imshow(Mn, cmap='viridis', vmin=0, vmax=1)
    ax.set_xlabel('policy action bin'); ax.set_ylabel('oracle action bin')
    ax.set_title(title); plt.colorbar(im, label='P(policy bin | oracle bin)')
    plt.tight_layout(); plt.show()

confusion(df_belief, 'Confusion: belief-optimal vs oracle')
if df_llm is not None:
    confusion(df_llm, 'Confusion: SFT LLM vs oracle')


## 5. Quote-vs-state curves

*Did the policy recover the BG structure?* Mean chosen offset bin as a function of (a) inventory and (b) regime, policy vs oracle. If the curves overlay, the LLM learned the inventory-skew and regime-skew shapes, not just the average.

In [ ]:
def quote_vs(df, by, title, oracle_col='a_oracle', pol_col='a_policy'):
    g = df.groupby(by)[[oracle_col, pol_col]].mean()
    fig, ax = plt.subplots(figsize=(7,4))
    ax.plot(g.index, g[oracle_col], 'o-', label='oracle', color='k')
    ax.plot(g.index, g[pol_col], 's--', label='policy', color='#e76f51')
    ax.set_xlabel(by); ax.set_ylabel('mean action bin'); ax.set_title(title); ax.legend()
    plt.tight_layout(); plt.show()
    return g

# inventory effect (bucket the oracle's inventory)
df_belief['q_bucket'] = df_belief['q_oracle'].round().clip(-5,5)
_ = quote_vs(df_belief, 'q_bucket', 'Offset vs inventory — belief-optimal vs oracle')
_ = quote_vs(df_belief.assign(regime_name=df_belief['regime'].map(REGIME)),
             'regime', 'Offset vs regime — belief-optimal vs oracle')
if df_llm is not None:
    df_llm['q_bucket'] = df_llm['q_oracle'].round().clip(-5,5)
    _ = quote_vs(df_llm, 'q_bucket', 'Offset vs inventory — SFT LLM vs oracle')
    _ = quote_vs(df_llm, 'regime', 'Offset vs regime — SFT LLM vs oracle')


## 6. Regime-inference quality

Since the regime is censored, the policy must infer it from flow. Action-correctness conditioned on belief-filter confidence tells whether the policy is doing **Bayesian inference** (correct when belief is confident) vs pattern-matching.

In [ ]:
def inference_quality(df, name):
    d = df.dropna(subset=['belief_true']).copy()
    d['conf_bin'] = pd.cut(d['belief_true'], bins=[0,0.3,0.5,0.7,0.9,1.01],
                           labels=['<.3','.3-.5','.5-.7','.7-.9','>.9'])
    g = d.groupby('conf_bin', observed=True)['match'].mean()
    fig, ax = plt.subplots(figsize=(7,4))
    ax.bar(g.index.astype(str), g.values, color='#264653')
    ax.set_xlabel('belief-filter posterior mass on TRUE regime')
    ax.set_ylabel('exact-match with oracle'); ax.set_title(f'Action correctness vs regime certainty — {name}')
    plt.tight_layout(); plt.show()
    return g

_ = inference_quality(df_belief, 'belief-optimal')
if df_llm is not None:
    _ = inference_quality(df_llm, 'SFT LLM')


---
### Notes for the final run
- Set `HAVE_LLM = True` and point `ADAPTER_PATH` at the fine-tuned adapter (GPU box).
- Bump `N_EVAL_CUSIPS` (e.g. 100) for publication-quality figures.
- All policies are rolled on **identical** held-out streams (same `SEED`), so comparisons are paired.
- Section 4's regime-conditioned match rate is the key diagnostic: it localizes *where* the LLM
  recovers the oracle (symmetric regimes, inventory control) vs where it struggles (imbalanced,
  hard-to-infer regimes) — exactly the structure the single gap number hides.